# SageMaker Notebook Demo: Bike Sharing Demand

- Dataset: [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (UCI)
- Regression: the target `cnt` is a count of rides per hour

Steps

1. setup notebook, install Python libraries
2. load `raw/hour.csv` from S3
3. engineer features, write to `featured/`
4. train model
   - baseline
   - deeper tree
5. evaluate, compare, and visualize
6. write the artifact to `model/`


## 1. Setup

`conda_python3` is a minimal kernel, so the libraries are installed first.

In [ ]:
%pip install -q scikit-learn pandas pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.


Get the bucket name from terraform:

```
terraform -chdir=infra output -raw data_bucket
```


In [ ]:
import boto3

REGION = "ca-central-1"

BUCKET = "mlops-sagemaker-dev-data-k35ss6"

RAW_KEY = "raw/hour.csv"
FEATURED_KEY = "featured/hour.parquet"
MODEL_KEY = "models/model.joblib"
FEATURES_KEY = "models/features.joblib"

s3 = boto3.client("s3", region_name=REGION)

s3.head_bucket(Bucket=BUCKET)
print(f"bucket reachable: {BUCKET}")

bucket reachable: sagemaker-notebook-dev-data-3opmm6


---

## 2. Load `raw/hour.csv`


In [ ]:
import io
import pandas as pd

obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"{len(df)} rows, {df.dteday.min()} -> {df.dteday.max()}")
df.head()

17379 rows, 2011-01-01 -> 2012-12-31


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


---

## 3. Engineer features

`casual + registered == cnt` in every row, so both leak the target.
`instant` is a row index, `dteday` is already covered by the calendar
columns. Drop all four.


In [ ]:
TARGET = "cnt"
DROP = ["instant", "dteday", "casual", "registered", TARGET]
FEATURES = [c for c in df.columns if c not in DROP]

leak = (df.casual + df.registered != df[TARGET]).sum()
print(f"rows where casual + registered != cnt: {leak}")
print(f"{len(FEATURES)} features: {FEATURES}")

rows where casual + registered != cnt: 0
12 features: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']


Write to `featured/`. Alice writes the shared prefix; bob writes his own
copy under `users/bob/`.

In [ ]:
featured = df[FEATURES + [TARGET]]

buf = io.BytesIO()
featured.to_parquet(buf, index=False)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, FEATURED_KEY)

print(f"{featured.shape[0]} rows x {featured.shape[1]} cols")
print(f"s3://{BUCKET}/{FEATURED_KEY}")

### Split by time

Train on 2011, test on 2012. A random split would let the model see
hours next to the ones it is scored on.

In [ ]:
train = featured[featured.yr == 0]
test = featured[featured.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")

train 8645 rows (2011)   test 8734 rows (2012)


---

## 4. Train

Two random forests on the same split:

- **baseline** -- `min_samples_leaf=5`
- **deeper** -- `min_samples_leaf=1`, fully grown trees


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


RUNS = {
    "baseline": {"n_estimators": 100, "min_samples_leaf": 5},
    "deeper": {"n_estimators": 100, "min_samples_leaf": 1},
}

models = {}
results = {}

for name, params in RUNS.items():
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_train, y_train)

    metrics = evaluate(y_test, model.predict(X_test))

    buf = io.BytesIO()
    joblib.dump(model, buf)
    metrics["size_mb"] = buf.tell() / 1024 / 1024

    models[name], results[name] = model, metrics
    print(f"{name:9} rmse={metrics['rmse']:6.1f}  mae={metrics['mae']:5.1f}  "
          f"r2={metrics['r2']:.3f}  {metrics['size_mb']:.1f}MB")

rmse=126.3  mae=89.4  r2=0.6342
(test mean count = 235)


---

## 5. Evaluate, compare, and visualize

An rmse of 126 means nothing on its own. Predicting the training mean
every hour is the floor to beat.


In [ ]:
floor = evaluate(y_test, np.full(len(y_test), y_train.mean()))

table = pd.DataFrame(results).T
table.loc["predict-the-mean"] = {**floor, "size_mb": 0.0}
table["vs_floor_%"] = (floor["rmse"] - table["rmse"]) / floor["rmse"] * 100

table.round(2)

  hr           0.651  #######################################
  temp         0.093  #####
  atemp        0.091  #####
  workingday   0.052  ###
  hum          0.030  #
  season       0.026  #


### Visualize - Baseline

In [ ]:
import matplotlib.pyplot as plt

BEST = "baseline"
model = models[BEST]

pred = model.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(f"Bike sharing demand -- 2012 holdout ({BEST})")

# what drives demand
ax = axes[0]
imp = pd.Series(model.feature_importances_, index=FEATURES).nlargest(8)
ax.barh(imp.index[::-1], imp.values[::-1])
ax.set_title("feature importance")

# predicted vs actual
ax = axes[1]
ax.scatter(y_test, pred, s=4, alpha=0.2)
lim = [0, max(y_test.max(), pred.max())]
ax.plot(lim, lim, ls="--", c="grey", lw=1)
ax.set_xlabel("actual")
ax.set_ylabel("predicted")
ax.set_title(f"predicted vs actual (r2={results[BEST]['r2']:.3f})")

# average day
ax = axes[2]
by_hour = pd.DataFrame({"hr": X_test.hr, "actual": y_test, "predicted": pred})
by_hour = by_hour.groupby("hr").mean()
ax.plot(by_hour.index, by_hour.actual, label="actual", lw=2)
ax.plot(by_hour.index, by_hour.predicted, label="predicted", lw=2, ls="--")
ax.set_xlabel("hour of day")
ax.set_ylabel("mean rides")
ax.set_title("average day")
ax.set_xticks(range(0, 24, 3))
ax.legend()

fig.tight_layout()
plt.show()

s3://sagemaker-notebook-dev-data-3opmm6/models/bike/model.joblib
s3://sagemaker-notebook-dev-data-3opmm6/models/bike/features.joblib


`hr` dominates. The model follows the 8am and 5-6pm commute peaks but
undershoots the tallest ones -- 2012 ridership grew past anything in the
2011 training data.

---

### 6. Write the artifact to `model/`

The baseline is the one worth keeping -- 0.9% worse on rmse, 6x smaller.
The feature list goes with it, because inference has to rebuild the
input columns in this exact order.

Alice writes the shared `model/`, which is what deployment serves. Bob
writes `users/bob/model/` -- he proposes a model, alice's stays
authoritative.

In [ ]:
def upload(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    buf.seek(0)
    s3.upload_fileobj(buf, BUCKET, key)
    print(f"s3://{BUCKET}/{key}")


upload(model, MODEL_KEY)
upload(FEATURES, FEATURES_KEY)

  predicted=   40.7  actual=   48
  predicted=   32.2  actual=   93
  predicted=   22.7  actual=   75
  predicted=   12.7  actual=   52
  predicted=    2.7  actual=    8
